In [22]:
# importa la librería 'requests' para hacer peticiones a la API pandas y numoy
import requests
import pandas as pd
import numpy as np
import mysql.connector


# Importamos la clave de API desde el archivo q nos dio en git keys.py
from keys import *

# define la ciudad y el país para poder buscar el clima
city = "Aberdeen"
country = "GB"

# hace la  petición a la API de OpenWeatherMap con los parámetros city, country y la clave de API
response = requests.get(f'http://api.openweathermap.org/data/2.5/forecast/?q={city},{country}&appid={OWM_key}&units=metric&lang=en')

In [ ]:
# convierte  la respuesta en formato JSON
data = response.json()

# imprimir el JSON 
print(data)

# extraer la lista de pronósticos del diccionario principal q contiene los datos
forecast_list = data.get('list', [])

# crea listas vacías para almacenar cada tipo de dato del pronóstico
times = []              # fecha y hora 
temperatures = []       # temperatura 
humidities = []         # porcentaje de humedad
weather_statuses = []   # estado general del clima 
wind_speeds = []        # velocidad del viento 
rain_volumes = []       # volumen de lluvia 
snow_volumes = []       # volumen de nieve 

# recorre cada entrada del pronóstico para extraer los datos
print(df.head())
print(df.dtypes)


for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))                                 
    temperatures.append(entry.get('main', {}).get('temp', np.nan))           
    humidities.append(entry.get('main', {}).get('humidity', np.nan))         
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan)) 
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))           
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))             
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))            

# crea un DataFrame con todos los datos extraídos  por columnas modificados de acuerdo a lo de github para que no haya error en sql
df = pd.DataFrame({
    'weather_datetime': times,
    'temperature': temperatures,
    'humidity': humidities,
    'weather_status': weather_statuses,
    'wind': wind_speeds,
    'rain_qty': rain_volumes,
    'snow': snow_volumes,
    'municipality_iso_country': 'Aberdeen City'
})

# muestra las primeras 5 filas del DataFrame
print(df.head())

import mysql.connector
print("Conexión lista")



df = df.replace({np.nan: None})

for _, row in df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO weather_data (
                weather_datetime, temperature, humidity, weather_status,
                wind, rain_qty, snow, municipality_iso_country
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            row['weather_datetime'], row['temperature'], row['humidity'],
            row['weather_status'], row['wind'], row['rain_qty'],
            row['snow'], row['municipality_iso_country']
        ))
    except Exception as e:
        print("Error en fila:", row)
        print("Detalle:", e)



{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1761782400, 'main': {'temp': 7.07, 'feels_like': 3.2, 'temp_min': 5.15, 'temp_max': 7.07, 'pressure': 995, 'sea_level': 995, 'grnd_level': 991, 'humidity': 91, 'temp_kf': 1.92}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'clouds': {'all': 75}, 'wind': {'speed': 6.92, 'deg': 319, 'gust': 12.37}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2025-10-30 00:00:00'}, {'dt': 1761793200, 'main': {'temp': 6.82, 'feels_like': 2.51, 'temp_min': 6.31, 'temp_max': 6.82, 'pressure': 996, 'sea_level': 996, 'grnd_level': 993, 'humidity': 86, 'temp_kf': 0.51}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'clouds': {'all': 69}, 'wind': {'speed': 8.1, 'deg': 342, 'gust': 11.76}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2025-10-30 03:00:00'}, {'dt': 1761804000, 'main': {'temp': 5.09, 'feels_like': 1.52, 'temp_min': 4.1

In [ ]:
# conexion a sql con el user y contraseña correspondientes
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='1309',
    database='gans'
)
conn.autocommit = True  #  evita bloqueos por transacciones abiertas (porque me aparecieron muchos errores de bloqueo)
cursor = conn.cursor()

# inserción de datos
for _, row in df.iterrows():
    print("Insertando fila:", row.to_dict())
    cursor.execute("""
        INSERT INTO weather_data (
            weather_datetime, temperature, humidity, weather_status,
            wind, rain_qty, snow, municipality_iso_country
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['weather_datetime'], row['temperature'], row['humidity'],
        row['weather_status'], row['wind'], row['rain_qty'],
        row['snow'], row['municipality_iso_country']
    ))

# cerramos conexión
cursor.close()
conn.close()

Insertando fila: {'weather_datetime': '2025-10-30 00:00:00', 'temperature': 7.07, 'humidity': 91, 'weather_status': 'Clouds', 'wind': 6.92, 'rain_qty': None, 'snow': None, 'municipality_iso_country': 'Aberdeen City'}
Insertando fila: {'weather_datetime': '2025-10-30 03:00:00', 'temperature': 6.82, 'humidity': 86, 'weather_status': 'Clouds', 'wind': 8.1, 'rain_qty': None, 'snow': None, 'municipality_iso_country': 'Aberdeen City'}
Insertando fila: {'weather_datetime': '2025-10-30 06:00:00', 'temperature': 5.09, 'humidity': 79, 'weather_status': 'Clouds', 'wind': 4.85, 'rain_qty': None, 'snow': None, 'municipality_iso_country': 'Aberdeen City'}
Insertando fila: {'weather_datetime': '2025-10-30 09:00:00', 'temperature': 5.96, 'humidity': 72, 'weather_status': 'Rain', 'wind': 2.44, 'rain_qty': 0.12, 'snow': None, 'municipality_iso_country': 'Aberdeen City'}
Insertando fila: {'weather_datetime': '2025-10-30 12:00:00', 'temperature': 10.62, 'humidity': 56, 'weather_status': 'Clouds', 'wind': 